# Day 15 ｜ Pandas：把「一次一个」变成「一次七千个」

今天用的数据是**你自己产品生成的**，不是网上下载的示例表。
先在终端（项目根目录）跑一次：

```
.venv/bin/python paipan/batch.py
```

看到 `写出 7305 行` 再回来往下跑。

## 为什么产品人要会 pandas

不是为了转数据分析岗。是因为这三件事只有 pandas 能便宜地做到：

1. **一眼看分布**。7305 行你读不完，但「日柱有 60 种、每种大约 122 天」这种话，
   一行 `value_counts()` 就能证实或推翻。分布不对，八成有事。
2. **把一条规则同时套到所有行上**。手工只能抽查 20 行，程序能一行不漏查 7305 行。
3. **喂给下一步**。接大模型做解读需要测试集，今天这份 CSV 就是明天的输入。

## 今天还有另一条主线（产品侧）

`app.py`（网页）、`core.py`（算法）、`batch.py`（批量）分家了。
原因：同一份算法写三遍，迟早分叉——页面显示 A、分析算出 B，这种 bug 最难查，
因为每一处**单独看都是对的**。规矩：**一处算法，多处使用**。
`core.py` 底部有 8 条边界自测，跑一下 `.venv/bin/python paipan/core.py` 就能看到。

## 学习方式

每格上面有一句「**先猜**」。先在心里给答案，再按 `Shift+Enter`。
猜错了才有收获，猜对了只是重复已知。

## 第 1 步 ｜ 拿到数据先看五个东西

`shape` / `dtypes` / `head` / `info` / `describe`。这五个动作各回答一个问题，
顺序不必死，但**读进文件第一件事就做**，别一上来就筛选。

先解决「文件在哪」。内核启动目录可能是项目根，也可能是 `notebooks/`，
所以别写死一条相对路径——列几个候选，找到哪个用哪个。

**先猜**：三处都没找到文件时，`CSV` 会是什么值？

In [25]:
from pathlib import Path

CANDS = [Path("paipan/data/charts.csv"),
         Path("../paipan/data/charts.csv"),
         Path.home() / "Documents/AI自学/ai-product-journey/paipan/data/charts.csv"]

CSV = next((c for c in CANDS if c.exists()), None)   # 第一个存在的；都没有就是 None
print("内核现在在:", Path.cwd())
print("找到数据文件:", CSV)

assert CSV is not None, "三处都没找到 charts.csv —— 回终端跑 .venv/bin/python paipan/batch.py"

内核现在在: /Users/makise057/Documents/AI自学/ai-product-journey/notebooks
找到数据文件: ../paipan/data/charts.csv


### ① shape：这块数据多大、多少栏

**先猜**：`(7305, ?)`，第二个数字在 `batch.py` 里数得出来。

In [26]:
import pandas as pd

# encoding="utf-8-sig"：batch.py 写的时候带了 BOM（为了 Excel 双击不乱码），读的时候说一声
# parse_dates=["date"]：把日期那栏变成【真日期类型】，不是字符串
df = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["date"])

print(df.shape)
print("列名：", list(df.columns))

(7305, 17)
列名： ['date', 'weekday', 'hour', 'minute', 'shichen', 'year_pillar', 'month_pillar', 'day_pillar', 'hour_pillar', 'day_master', 'day_master_wx', 'mu', 'huo', 'tu', 'jin', 'shui', 'early_zi']


### ② dtypes：每一栏到底是什么类型

类型决定你能对它做什么。文本栏不能加减，日期不解析就只能比字符。

**先猜**：17 栏里有几栏是数字、几栏是文本、几栏是日期？

In [27]:
df.dtypes

date             datetime64[us]
weekday                   int64
hour                      int64
minute                    int64
shichen                     str
year_pillar                 str
month_pillar                str
day_pillar                  str
hour_pillar                 str
day_master                  str
day_master_wx               str
mu                        int64
huo                       int64
tu                        int64
jin                       int64
shui                      int64
early_zi                  int64
dtype: object

### ③ head：前几行实际长什么样

`head()` 默认 5 行，`head(3)` 给 3 行。它的用处是**核对读取有没有错位**——
列串了、日期变字符串、中文变乱码，扫一眼就发现。

In [28]:
df.head(3)

,date,weekday,hour,minute,shichen,year_pillar,month_pillar,day_pillar,hour_pillar,day_master,day_master_wx,mu,huo,tu,jin,shui,early_zi
0,2000-01-01,5,0,0,子,己卯,丙子,戊午,壬子,戊,土,1,2,2,0,3,0
1,2000-01-02,6,7,13,辰,己卯,丙子,己未,戊辰,己,土,1,1,5,0,1,0
2,2000-01-03,0,14,26,未,己卯,丙子,庚申,癸未,庚,金,1,1,2,2,2,0


### ④ info：有多少行没值、各栏什么类型

`info()` 一次把 shape、每栏非空计数、类型给全。**看缺失值就看这一屏**。

**先猜**：这份数据会有缺失值吗？

In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7305 entries, 0 to 7304
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           7305 non-null   datetime64[us]
 1   weekday        7305 non-null   int64         
 2   hour           7305 non-null   int64         
 3   minute         7305 non-null   int64         
 4   shichen        7305 non-null   str           
 5   year_pillar    7305 non-null   str           
 6   month_pillar   7305 non-null   str           
 7   day_pillar     7305 non-null   str           
 8   hour_pillar    7305 non-null   str           
 9   day_master     7305 non-null   str           
 10  day_master_wx  7305 non-null   str           
 11  mu             7305 non-null   int64         
 12  huo            7305 non-null   int64         
 13  tu             7305 non-null   int64         
 14  jin            7305 non-null   int64         
 15  shui           7305 non-null   i

### ⑤ describe：数值栏的一屏体检

count / mean / min / 25% / 50%（中位数）/ 75% / max / std。

**先猜两件事**：
- 文本栏（`day_pillar` 这些）会不会出现在报告里？
- 日期栏的 `std` 能算出来吗？

In [30]:
df.describe()

,date,weekday,hour,minute,mu,huo,tu,jin,shui,early_zi
count,7305,7305.000000,7305.000000,7305.000000,7305.000000,7305.000000,7305.000000,7305.000000,7305.000000,7305.000000
mean,2009-12-31 00:00:00,3.000000,11.497331,29.494867,1.402875,1.504449,2.149897,1.502943,1.439836,0.041615
min,2000-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2004-12-31 00:00:00,1.000000,5.000000,14.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
50%,2009-12-31 00:00:00,3.000000,11.000000,29.000000,1.000000,1.000000,2.000000,1.000000,1.000000,0.000000
75%,2014-12-31 00:00:00,5.000000,17.000000,44.000000,2.000000,2.000000,3.000000,2.000000,2.000000,0.000000
max,2019-12-31 00:00:00,6.000000,23.000000,59.000000,5.000000,6.000000,7.000000,4.000000,6.000000,1.000000
std,NaN,2.000479,6.923046,17.319465,1.070575,1.069984,1.299953,1.044871,1.084202,0.199722


看到两个细节：**文本栏根本不在里面**，日期栏的 `std` 是 `NaN`。
`describe()` 默认只统计「能算平均数的栏」，文本栏要点名才给：

In [31]:
df.describe(include=["str"])   # pandas 3 里文本栏的类型叫 str（老版本叫 object）

,shichen,year_pillar,month_pillar,day_pillar,hour_pillar,day_master,day_master_wx
count,7305,7305,7305,7305,7305,7305,7305
unique,12,21,60,60,60,10,5
top,辰,庚辰,甲午,甲子,壬子,戊,金
freq,610,366,127,183,122,792,1463


**读 `top` 和 `freq` 这两行**：`top` = 出现次数最多的值，`freq` = 它出现几次。

先别急着看答案，想一遍：`day_pillar` 的 `freq` 应该是多少？
（提示：一天一个柱，60 天一轮回，7305 天 ÷ 60 = 121.75）

In [32]:
print("day_pillar 的 top / freq：", df.day_pillar.value_counts().idxmax(),
      int(df.day_pillar.value_counts().max()))
print("如果它是 121 或 122，说明日柱转得很匀；差很远就有故事查。")

day_pillar 的 top / freq： 甲子 183
如果它是 121 或 122，说明日柱转得很匀；差很远就有故事查。


---

### 五个动作各自回答什么

| 动作 | 回答的问题 | 能发现什么 |
|---|---|---|
| `shape` | 我拿到多大一块？ | 行数不对＝读错文件/漏读行 |
| `dtypes` | 每栏是什么类型？ | 日期是字符串＝后面比较全错 |
| `head` | 数据实际长什么样？ | 列错位、乱码、单位不对 |
| `info` | 有多少没值？ | 缺失值、脏数据 |
| `describe` | 分布正不正常？ | 异常值、比例失衡 |

⏸ **暂停点 1**：这五格跑完、能说出各自用途，第一个目标就达成了。

## 第 2 步 ｜ 选栏、加栏

一栏和几栏，拿出来的东西**不是同一种对象**：

```python
df["day_pillar"]                    # → Series（一维，带标签的列）
df[["day_pillar", "hour_pillar"]]   # → DataFrame（二维，还是张小表）
```

一个方括号＝取一栏；**两个方括号**＝「我要几栏，列成列表给我」。
这个区别决定后面能不能 `.str`、能不能 `.sum(axis=1)`，搞混了报错会很绕。

In [9]:
one = df["day_pillar"]
two = df[["day_pillar", "hour_pillar"]]
print(type(one), tuple(one.shape))
print(type(two), tuple(two.shape))

<class 'pandas.Series'> (7305,)
<class 'pandas.DataFrame'> (7305, 2)


### 取日主：天干就是柱子的第一个字

`日主`（日柱天干）是八字里最要紧的一个字，整个盘围绕它算。
文本栏想用字符串方法，前面加 `.str`。

**先猜**：`df["day_pillar"].str[0]` 和 `batch.py` 早就算好存进 CSV 的 `day_master` 栏，会完全一样吗？

In [10]:
computed = df["day_pillar"].str[0]
saved = df["day_master"]
same = (computed == saved)
print("两栏完全一致吗:", bool(same.all()))
print("不一致的行数:", int((~same).sum()))      # 0 才算对

两栏完全一致吗: True
不一致的行数: 0


两条路**互相独立**：一条是 Python 现在切字符串，一条是当初写 CSV 时存下来的。
两条不同的路得到同一个结果，叫**交叉验证**（错题 #36、#44），比单看一条强得多。
不一致的话说明 CSV 和数据不同源，别往下算了。

### 加一栏：把算出来的结果当新栏

`df["新列名"] = ...` 就是加栏（已存在则覆盖）。
按 Day 13 定的规矩（地支只算本气、不含藏干），**每行五行个数必须等于 8**：4 个天干 + 4 个地支。

In [11]:
WX5 = ["mu", "huo", "tu", "jin", "shui"]
df["wx_total"] = df[WX5].sum(axis=1)   # axis=1 = 横着加（每行内部）；axis=0 竖着加（默认）
print(df[["day_pillar", "mu", "huo", "tu", "jin", "shui", "wx_total"]].head(3))
print()
print("不等于 8 的行数:", int((df.wx_total != 8).sum()))

  day_pillar  mu  huo  tu  jin  shui  wx_total
0         戊午   1    2   2    0     3         8
1         己未   1    1   5    0     1         8
2         庚申   1    1   2    2     2         8

不等于 8 的行数: 0


`axis=1` 是「沿着栏的方向走」＝每行内部把 5 个数加起来。记法：**0 竖 1 横**。

### 筛选：条件写在方括号里

`early_zi` 这栏记的是「这条走了 23 点后日柱进次日」的规则。

**先猜**：7305 天里 23 点会出现多少次？

In [13]:
night = df[df.hour >= 23]
print("hour >= 23 的行数:", len(night))
print("early_zi == 1 的行数:", int(df.early_zi.sum()))
print("两个数相等吗:", len(night) == int(df.early_zi.sum()))

hour >= 23 的行数: 304
early_zi == 1 的行数: 304
两个数相等吗: True


两个数该一样：`early_zi` 就是按 `hour >= 23` 写的。
但这里是**两条独立路径算同一个数**（一条读现成栏，一条重新比 `hour`）。
不一致就说明 CSV 内部自相矛盾。

多条件用 `&`（且）、`|`（或）、`~`（非），**每个条件各自加括号**：

```python
df[(df.hour >= 23) & (df.weekday == 5)]   # 对
df[df.hour >= 23 & df.weekday == 5]       # 错，而且报错信息看不懂
```

**先猜**：为什么不能用 Python 的 `and` / `or`？

In [14]:
mask = (df.hour >= 23) & (df.weekday == 5)
print("23 点又正好周六的行数:", int(mask.sum()))
print("23 点或 0 点的行数:", int(((df.hour == 23) | (df.hour == 0)).sum()))
df[mask][["date", "hour", "shichen", "day_pillar", "hour_pillar"]]

23 点又正好周六的行数: 43
23 点或 0 点的行数: 609


,date,hour,shichen,day_pillar,hour_pillar
161,2000-06-10,23,子,庚子,丙子
329,2000-11-25,23,子,戊子,壬子
497,2001-05-12,23,子,丙子,戊子
665,2001-10-27,23,子,甲子,甲子
833,2002-04-13,23,子,壬子,庚子
1001,2002-09-28,23,子,庚子,丙子
1169,2003-03-15,23,子,戊子,壬子
1337,2003-08-30,23,子,丙子,戊子
1505,2004-02-14,23,子,甲子,甲子
1673,2004-07-31,23,子,壬子,庚子


不能用 `and`：`and` 要判断「整个东西真不真」，而你给的 `df.hour >= 23` 是
**一整列** 7305 个真假值，Python 不知道你想怎么合并，于是报
`The truth value of a Series is ambiguous`。`&` 是**逐元素**比，各比各的。

顺便看上面那张小表：0 点和 23 点**同属子时**，但 0 点日柱不进次日、23 点进。
这就是「早子时派」的争议点——时辰同名，日柱归属不同。
不是 bug，是你 Day 13 选的规矩。但你得能在数据里**看见**它。

⏸ **暂停点 2**：到这里「取数、加栏、筛选」三个基本动作都碰过了。

## 第 3 步 ｜ `loc` 和 `iloc`：一个按标签，一个按位置

- `iloc` = **i**nteger **loc**ation，按第几行第几列（永远给数字）
- `loc` = 按**标签**（行标签、列名，也可以是条件）

先看一个反直觉的差别。**先猜**：两个切片各几行？

In [15]:
print("iloc[0:5] →", len(df.iloc[0:5]), "行")
print("loc[0:5]   →", len(df.loc[0:5]), "行")
df.loc[0:5, ["date", "day_pillar"]]

iloc[0:5] → 5 行
loc[0:5]   → 6 行


,date,day_pillar
0,2000-01-01,戊午
1,2000-01-02,己未
2,2000-01-03,庚申
3,2000-01-04,辛酉
4,2000-01-05,壬戌
5,2000-01-06,癸亥


**`loc` 切片两头都含**，所以 `0:5` 是 6 行；`iloc` 跟普通 Python 切片一样含头不含尾，是 5 行。
这是 pandas 最有名的坑之一：多出来那一行**不报错**，只让你的数「莫名大了 1」。
不报错的错最贵（错题 #35）。

`loc` 第一个参数选行（可放条件），第二个选栏：

In [16]:
sub = df.loc[df.hour == 23, ["date", "day_pillar", "hour_pillar"]]
print(sub.head())
print("行数:", len(sub))

          date day_pillar hour_pillar
17  2000-01-18         丙子          戊子
41  2000-02-11         庚子          丙子
65  2000-03-06         甲子          甲子
89  2000-03-30         戊子          壬子
113 2000-04-23         壬子          庚子
行数: 304


把日期设成索引，就能像查字典一样查某一天，还能切片：

In [17]:
by_date = df.set_index("date")
by_date.loc["2000-02-02":"2000-02-06", ["year_pillar", "month_pillar", "day_pillar"]]

,year_pillar,month_pillar,day_pillar
date,,,
2000-02-02,己卯,丁丑,庚寅
2000-02-03,己卯,丁丑,辛卯
2000-02-04,庚辰,戊寅,壬辰
2000-02-05,庚辰,戊寅,癸巳
2000-02-06,庚辰,戊寅,甲午


年柱在 2000-02-04 从 `己卯` 跳到 `庚辰`——立春分年，精确到分钟（那年立春是 20:41）。
而**日柱一路顺走**（辛卯→壬辰→癸巳），换年柱没有波及它。
这正是 `core.py` 里那条回归靶子（Day 14 的教训：整体 +1 天会连时柱一起推走）。

`set_index` 之后 `loc["2000-02-02":"2000-02-06"]` 是**闭区间**，两端都含。

## 第 4 步 ｜ 分布 = 免费的正确性检查器（今天的主戏）

### ① 先看日柱匀不匀

六十甲子一天一个，永不重复、永不跳过。7305 天 = 60 × 121 + 45，
所以**如果没人搬动**，应该 45 种出现 122 次、15 种出现 121 次。
这个 122/121 是我用除法手算的，**不是从 pandas 的输出抄的**（错题 #24、#38）。

**先猜**：实测会落在 121/122 上吗？

In [18]:
vc = df.day_pillar.value_counts()          # 每个值出现几次，从多到少排
print("日柱种类数:", len(vc))
print("最多的前 5:", dict(vc.head(5)))
print("最少的后 5:", dict(vc.tail(5)))
print()
print("次数 → 有多少种日柱是这个次数:")
print(vc.value_counts().sort_index().to_string())

日柱种类数: 60
最多的前 5: {'甲子': np.int64(183), '丙子': np.int64(183), '戊子': np.int64(183), '庚子': np.int64(183), '壬子': np.int64(181)}
最少的后 5: {'癸亥': np.int64(61), '丁亥': np.int64(61), '辛亥': np.int64(61), '乙亥': np.int64(61), '己亥': np.int64(61)}

次数 → 有多少种日柱是这个次数:
count
61      5
121    13
122    37
181     1
183     4


**这里就不对劲了**。大部分日柱在 121/122 附近（正常），
但有 **5 种特别高、5 种特别低**，而且低到 61、高到 183——
61 + 122 ≈ 183，看起来像是有一批行被**成建制地搬走了**。

先别下结论说「引擎有 bug」。**报错/异常的怀疑要先用独立验算排除掉**。

### ② 独立验算：不用命理库，把整条日柱重算一遍

六十甲子严格按天推进，只要知道**任何一个锚点日期**，全表都能推出来。
锚点：1949-10-01 = **甲子日**（开国大典，公开可查；`lunar_python` 和 `cnlunar` 两个库也都给甲子）。

于是这栏的期望值**完全由日期算术产生，一步都没经过生成数据的代码**。

In [19]:
from datetime import date

GAN = "甲乙丙丁戊己庚辛壬癸"
ZHI = "子丑寅卯辰巳午未申酉戌亥"
CYCLE = [GAN[i % 10] + ZHI[i % 12] for i in range(60)]    # 自己拼六十甲子表
ANCHOR = date(1949, 10, 1)                                 # 甲子日

offset = (df.date.dt.date - ANCHOR).map(lambda d: d.days)  # 每天距锚点几天
# 走了早子时规则的行，日柱再往后一位；early_zi 正好是 0/1，直接拿来加
df["expected_day"] = [CYCLE[(o + e) % 60] for o, e in zip(offset, df.early_zi)]

mismatch = df[df.expected_day != df.day_pillar]
print("不一致的行数:", len(mismatch), "／ 共", len(df))
print("结论：", "算法没错——那 10 种跑偏不是算错，是【样本被搬动】造成的"
      if len(mismatch) == 0 else f"有 {len(mismatch)} 行对不上，这是真 bug，查 core.py")
mismatch.head()

不一致的行数: 0 ／ 共 7305
结论： 算法没错——那 10 种跑偏不是算错，是【样本被搬动】造成的


,date,weekday,hour,minute,shichen,year_pillar,month_pillar,day_pillar,hour_pillar,day_master,day_master_wx,mu,huo,tu,jin,shui,early_zi,wx_total,expected_day


**这一格是今天最值钱的一格。** 7305 行、一条没错，而且这个「没错」
是靠一个完全独立的公式对出来的。

所以现在事实很清楚，两条都不肯让路：
- ① **分布歪了**（5 种 183 次、5 种 61 次）
- ② **算法没错**（跟纯算术完全一致）

两个都对的时候，第三个东西在骗你——**取样**。往下看那 10 种是谁。

### ③ 被搬走的是谁？

In [20]:
high = [p for p in vc.index if vc[p] > 150]
low = [p for p in vc.index if vc[p] < 90]
print("偏多的 5 种:", high)
print("偏少的 5 种:", low)
print()
print("它们在六十甲子表上的位置:", [CYCLE.index(p) for p in sorted(high, key=CYCLE.index)],
      [CYCLE.index(p) for p in sorted(low, key=CYCLE.index)])
print()
print("看字形：偏多的一边天干是", sorted({p[0] for p in high}), "地支是", sorted({p[1] for p in high}))
print("        偏少的一边天干是", sorted({p[0] for p in low}), "地支是", sorted({p[1] for p in low}))

偏多的 5 种: ['甲子', '丙子', '戊子', '庚子', '壬子']
偏少的 5 种: ['癸亥', '丁亥', '辛亥', '乙亥', '己亥']

它们在六十甲子表上的位置: [0, 12, 24, 36, 48] [11, 23, 35, 47, 59]

看字形：偏多的一边天干是 ['丙', '壬', '庚', '戊', '甲'] 地支是 ['子']
        偏少的一边天干是 ['丁', '乙', '己', '癸', '辛'] 地支是 ['亥']


**偏少的 5 种全是 `亥` 日，偏多的 5 种全是 `子` 日**，天干一边全阴、一边全阳。
也就是说：每一次「23 点后日柱进次日」，都是把一个**亥日**搬成了**子日**。

这不是巧合。亥是十二地支最后一个，子是最前面一个——`+1 天` 正好把它们连起来。
问题变成：**为什么走规则的 304 行，全都是亥日？**

回 `batch.py` 看那行取样代码：

```python
dt = d.replace(hour=(i * 7) % 24, minute=(i * 13) % 60)
```

只有 `(i * 7) % 24 == 23` 的那些天才算 23 点。解出来是 `i ≡ 17 (mod 24)`——
每 24 天一次。而**地支的周期是 12 天，24 正好是 12 的倍数**。
所以这 304 行在地支上永远落在同一个位置：亥。

In [21]:
shifted = df[df.early_zi == 1].copy()
shifted["本来的地支"] = [CYCLE[o % 60][1] for o in (shifted.date.dt.date - ANCHOR).map(lambda d: d.days)]
print("走规则的 304 行，原本的地支分布：")
print(shifted["本来的地支"].value_counts().to_string())
print()
print("如果取样公平，12 个地支该各 25 次左右（304 / 12 ≈ 25）。")
print("现在有几个地支是有数的？其余 11 个是不是全都不存在？")

走规则的 304 行，原本的地支分布：
本来的地支
亥    304

如果取样公平，12 个地支该各 25 次左右（304 / 12 ≈ 25）。
现在有几个地支是有数的？其余 11 个是不是全都不存在？


一个地支吃掉全部 304 行，其余 11 个是 0。**这就是取样偏差的铁证。**

再看它的后果——日主（天干）本该十个数几乎相等，因为被搬的全是阴干日：

In [22]:
YANG = {GAN[i] for i in (0, 2, 4, 6, 8)}      # 甲丙戊庚壬
YIN = {GAN[i] for i in (1, 3, 5, 7, 9)}        # 乙丁己辛癸

yang_now = int(df.day_master.isin(YANG).sum())
yin_now = int(df.day_master.isin(YIN).sum())
print("实测 阳干日 / 阴干日 =", yang_now, "/", yin_now, "差", abs(yang_now - yin_now))
print()
print("手算闭环：7305 天里本来 阳 3653 / 阴 3652（一天一换）。")
print("304 行从阴干（亥）搬到阳干（子）→ 阳 3653+304 =", 3653 + 304, "，阴 3652-304 =", 3652 - 304)
print("对得上吗:", (3653 + 304 == yang_now) and (3652 - 304 == yin_now))

实测 阳干日 / 阴干日 = 3957 / 3348 差 609

手算闭环：7305 天里本来 阳 3653 / 阴 3652（一天一换）。
304 行从阴干（亥）搬到阳干（子）→ 阳 3653+304 = 3957 ，阴 3652-304 = 3348
对得上吗: True


数字精确对上，说明整条解释链闭合了：
**取样周期（24）与地支周期（12）咬合 → 搬动单向 → 阴阳干失衡 609（= 304 × 2 + 1）。**

**结论：这是取样设计的缺陷，不是排盘算法的缺陷。** 分清这两种「不均」就是产品人的数据敏感度。

说清楚一句：这个坑是我写 `batch.py` 时**无意间**埋的，不是给你设的考题。
我当时只想到「7 和 24 互质能把 24 个时刻扫全」，没看出 24 是 12 的倍数会跟地支锁死。
**你在数据里发现的这件事，比我写代码时想到的更细。** 这就是 value_counts 的价值。

### ④ 另一种不均：结构上本来就该这样

五行总个数。看之前先想：12 个地支里属土的有几个？

In [23]:
wx_sum = df[WX5].sum()
print("五行总量（7305 行 × 8 字 = 58440 个字）:")
print(wx_sum.sort_values(ascending=False).to_string())

五行总量（7305 行 × 8 字 = 58440 个字）:
tu      15705
huo     10990
jin     10979
shui    10518
mu      10248


**土明显最多，但这不是 bug**：
- 地支 12 个：子水 丑土 寅木 卯木 辰土 巳火 午火 未土 申金 酉金 戌土 亥水
  → 土占 **4 个**（辰戌丑未），其余五行各 2 个
- 天干 10 个：五行各 2 个 → 均匀

期望值先手算，再跟实测比：

In [24]:
total = int(df.shape[0]) * 8            # 8 是每行字数，不是行数
gan_each = total / 2 / 5                 # 天干部分：每个五行应得
zhi_each = total / 2 * 2 / 12            # 地支部分：普通五行（2/12）
zhi_tu = total / 2 * 4 / 12              # 地支部分：土（4/12）
print("手算期望 → 普通五行各约", round(gan_each + zhi_each), "，土约", round(gan_each + zhi_tu))
print("实测    →", {k: int(v) for k, v in wx_sum.items()})
print()
print("土是唯一高出期望一大截的吗？是的话，那是【结构】，不是【数据质量问题】。")

手算期望 → 普通五行各约 10714 ，土约 15584
实测    → {'mu': 10248, 'huo': 10990, 'tu': 15705, 'jin': 10979, 'shui': 10518}

土是唯一高出期望一大截的吗？是的话，那是【结构】，不是【数据质量问题】。


---

## 第 5 步 ｜ 动手修那个取样偏差（今天唯一要你改的代码）

**改的是 `paipan/batch.py`，不是这个 notebook 的格子。**

打开 `paipan/batch.py`，在 `make_rows()` 里找到这一行：

```python
dt = (d.replace(hour=(i * 7) % 24, minute=(i * 13) % 60))
```

换成下面这两行；顺手在文件顶部 `import csv` 那行下面加一行 `import random`：

In [ ]:
# ↓↓↓ 下面的代码是贴进 paipan/batch.py 的 make_rows() 里的，不是让你在这里运行 ↓↓↓
# import random                    # ← 加到文件顶部 import csv 的下面
# rng = random.Random(20261102)     # ← 加在 make_rows() 里 rows = [] 的下面
# dt = (d.replace(hour=rng.randrange(24), minute=rng.randrange(60)))
# ↑ 定种子的随机数：既把 24 个时刻铺满，又不跟日期周期咬合，而且每次跑结果一样（可复现）
print("这一格只是给你复制代码用的。它不改动任何文件，运行也无害。")

**为什么用带种子的随机，而不是直接 `random.randrange(24)`？**
不写种子，每次生成的 CSV 都不一样，你没法回答「这次比上次好是因为我改了代码，
还是因为骰子重掷了一次」。**可复现 = 能比较**，这是拿数据做决策的前提。

改完存盘（Cmd+S），回终端跑：

```
.venv/bin/python paipan/batch.py
```

然后**从第 1 步那格重新往下跑**（内核里还是旧 CSV——Python 不会热重载，错题 #42）。
懒得整跑就单跑下面这格：

In [33]:
fresh = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["date"])
fvc = fresh.day_pillar.value_counts()
f_yang = int(fresh.day_master.isin(YANG).sum())
f_yin = int(fresh.day_master.isin(YIN).sum())
print("日柱次数  min / max =", int(fvc.min()), "/", int(fvc.max()), "（修之前 61 / 183）")
print("次数分布：", fvc.value_counts().sort_index().to_dict())
print()
print("阳干 / 阴干 =", f_yang, "/", f_yin, "差", abs(f_yang - f_yin), "（修之前差 609）")
print("走早子时的行数:", int(fresh.early_zi.sum()))

日柱次数  min / max = 115 / 128 （修之前 61 / 183）
次数分布： {115: 1, 116: 4, 118: 4, 119: 5, 120: 9, 121: 3, 122: 7, 123: 10, 124: 7, 125: 1, 126: 6, 127: 1, 128: 2}

阳干 / 阴干 = 3689 / 3616 差 73 （修之前差 609）
走早子时的行数: 288


**预定靶子**（我用同一个种子跑出来的数，你的应该一模一样）：

| | 修之前 | 修之后 |
|---|---|---|
| 日柱次数 min / max | 61 / 183 | **115 / 128** |
| 阳干 / 阴干 | 3957 / 3348（差 609） | **3689 / 3616（差 73）** |
| 走早子时规则的行数 | 304（每 24 天准点一次） | **288**（不再准点） |
| 被搬走那批原本的地支 | 亥 304（全包） | **12 个都有，各 11～35 次** |

数不一样先自查三件事：种子是不是 `20261102`；`rng` 是不是建在循环**外面**
（放进去＝每行重新初始化，每行都摇出同一个数）；`batch.py` 是不是真存盘并重跑了。

**判断标准从来不是「绝对均匀」，而是：**
> 偏离能被你的规则解释、方向双向、量级对得上 → 正常；
> 单向、量级正好等于某一批的行数 → 取样有问题。

剩下那 73 的差是规则的正常后果：现在搬动是双向的（有的阳搬成阴、有的阴搬成阳），
进出互相抵消一大半，抵消不完就剩几十。重跑第 4 步③那格能自己确认。

---

## 今天的收尾（三件事）

**1. `NOTES.md` 写三行**（Day 15）：
- 今天学了什么：别写「学了 pandas」，写你**具体会做什么了**
- 卡在哪：真实的那个点（写「不懂」等于没写）
- 明天要做什么

**2. Git 提交**（存盘 ≠ 提交，错题 #18）。终端里：

```
cd ~/Documents/AI自学/ai-product-journey
git add paipan/core.py paipan/app.py paipan/batch.py notebooks/15_pandas.ipynb NOTES.md
git status
```

看一眼 `git status` 里 `charts.csv` 有没有出现（不该出现，被 `.gitignore` 的 `*.csv` 挡住）。
再：

```
git commit -m "Day 15：排盘算法抽到 core.py，batch.py 批量取样，pandas 查出取样偏差"
git push origin main
```

提交信息请照你**实际改了什么**自己念一遍再写，别直接抄我的（错题 #20）。

**3. 错题集**：今天凡是「我以为……结果不是」的，直接说给我。
我这边已经攒了几条自己的（那个取样偏差就是其一），收工时一起补进表里。

## 加餐（选做，不做不可惜）

- 把 `weekday` 变成中文（周一…周日）加一栏：用 `.map()` 传一个字典 `{0: "周一", ...}`
- 数哪些日子五行里**一个金都没有**，看它们有什么共同点
- 用 `.value_counts()` 看 `month_pillar`：60 种月柱匀不匀？如果不匀，
  **先判断是算法问题还是取样问题**，再动手——这就是今天练的那招